**Create Schema and volume first**

In [0]:
-- Check which catalog is active
SELECT current_catalog();

-- Create and select the schema
CREATE SCHEMA IF NOT EXISTS testing;
USE SCHEMA testing;

-- Create a managed volume inside the schema
CREATE VOLUME IF NOT EXISTS testing;

-- Verify
DESCRIBE VOLUME testing;

Then saka mo isasalpak ung customer data mo

In [0]:
USE SCHEMA testing;

-- Existing trusted customer table
CREATE OR REPLACE TABLE customers (
    customer_id INT,
    name STRING,
    city STRING
)
USING DELTA;

INSERT INTO customers VALUES
    (101, 'Ana',   'Manila'),
    (102, 'Ben',   'Cebu'),
    (103, 'Carlo', 'Davao');


-- Today's incoming batch, including a duplicate
CREATE OR REPLACE TABLE customers_batch (
    customer_id INT,
    name STRING,
    city STRING
)
USING DELTA;

INSERT INTO customers_batch VALUES
    (101, 'Ana',  'Makati'),
    (102, 'Ben',  'Cebu'),
    (104, 'Dana', 'Iloilo'),
    (104, 'Dana', 'Iloilo');

After non papashow mo ung laman ng customer table mo

In [0]:
SELECT * FROM customers ORDER BY customer_id;
SELECT * FROM customers_batch ORDER BY customer_id;

As u can see may duplicates so ayaw natin un

In [0]:
MERGE INTO testing.customers AS target
USING (
    SELECT DISTINCT
        customer_id,
        name,
        city
    FROM testing.customers_batch
) AS source
ON target.customer_id = source.customer_id

-- Update only when something changed
WHEN MATCHED
     AND NOT (
         target.name <=> source.name
         AND target.city <=> source.city
     )
THEN UPDATE SET
    target.name = source.name,
    target.city = source.city

-- Insert new customers
WHEN NOT MATCHED
THEN INSERT (
    customer_id,
    name,
    city
)
VALUES (
    source.customer_id,
    source.name,
    source.city
);

so ngaun ay distinct na sila

In [0]:
SELECT *
FROM testing.customers
ORDER BY customer_id;

so dito na papasok ung new batch 

In [0]:
TRUNCATE TABLE testing.customers_batch;

INSERT INTO testing.customers_batch VALUES
    (101, 'Ana',  'Makati'),  -- unchanged
    (102, 'Ben',  'Bohol'),   -- changed
    (105, 'Ella', 'Pasig'),   -- new
    (105, 'Ella', 'Pasig');   -- duplicate

ok nag merge ako dito ng with new data and old data

In [0]:
MERGE INTO testing.customers AS target
USING (
    SELECT DISTINCT customer_id, name, city
    FROM testing.customers_batch
) AS source
ON target.customer_id = source.customer_id

WHEN MATCHED
     AND NOT (
         target.name <=> source.name
         AND target.city <=> source.city
     )
THEN UPDATE SET
    target.name = source.name,
    target.city = source.city

WHEN NOT MATCHED
THEN INSERT (customer_id, name, city)
VALUES (source.customer_id, source.name, source.city);

ok na nadagdagan na

In [0]:
SELECT *
FROM testing.customers
ORDER BY customer_id;